In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import numpy as np
import copy

# ==========================================
# 1. CONFIGURACIÓN Y REPRODUCIBILIDAD
# ==========================================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 2. GENERACIÓN Y PREPARACIÓN DE DATOS
# ==========================================
def generate_communications_data(num_samples: int, noise_variance: float):
    symbols = np.random.choice([-2.0, 2.0], size=num_samples)
    noise_std = np.sqrt(noise_variance)
    noise = np.random.normal(0, noise_std, size=num_samples)
    y = symbols + noise
    t = np.where(symbols == 2.0, 1.0, 0.0)
    return y.astype(np.float32), t.astype(np.float32)

def prepare_dataloaders(y, t, batch_size=64):
    y_train_val, y_test, t_train_val, t_test = train_test_split(y, t, test_size=0.15, random_state=42)
    y_train, y_val, t_train, t_val = train_test_split(y_train_val, t_train_val, test_size=0.1765, random_state=42)

    # Extraemos media y desviación estándar de entrenamiento
    y_mean, y_std = y_train.mean(), y_train.std()
    y_std = y_std if y_std > 0 else 1.0

    # Estandarización
    y_train = (y_train - y_mean) / y_std
    y_val = (y_val - y_mean) / y_std
    y_test = (y_test - y_mean) / y_std

    train_dataset = TensorDataset(torch.tensor(y_train).unsqueeze(1), torch.tensor(t_train).unsqueeze(1))
    val_dataset = TensorDataset(torch.tensor(y_val).unsqueeze(1), torch.tensor(t_val).unsqueeze(1))
    test_dataset = TensorDataset(torch.tensor(y_test).unsqueeze(1), torch.tensor(t_test).unsqueeze(1))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # RETORNAMOS LA MEDIA Y DESVIACIÓN ESTÁNDAR PARA USARLAS EN INFERENCIA
    return train_loader, val_loader, test_loader, float(y_mean), float(y_std)

# ==========================================
# 3. ARQUITECTURA DE LA RED
# ==========================================
class DetectorMLP(nn.Module):
    def __init__(self, hidden_neurons=16):
        super(DetectorMLP, self).__init__()
        self.fc1 = nn.Linear(1, hidden_neurons)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons)
        self.relu2 = nn.ReLU()
        self.fc_out = nn.Linear(hidden_neurons, 1)
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        z1 = self.fc1(x)
        h1 = self.relu1(z1)
        z2 = self.fc2(h1)
        h2 = self.relu2(z2)
        z_out = self.fc_out(h2)
        return self.sigmoid(z_out)

# ==========================================
# 4. ENTRENAMIENTO Y EVALUACIÓN
# ==========================================
def train_model(model, train_loader, val_loader, epochs=50, learning_rate=0.001, device='cpu'):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for y_batch, t_batch in train_loader:
            y_batch, t_batch = y_batch.to(device), t_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(y_batch), t_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * y_batch.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for y_val, t_val in val_loader:
                y_val, t_val = y_val.to(device), t_val.to(device)
                val_loss += criterion(model(y_val), t_val).item() * y_val.size(0)

        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_weights)
    return model

# ==========================================
# 5. PRODUCCIÓN: INFERENCIA EN TIEMPO REAL
# ==========================================
def detectar_simbolo_en_tiempo_real(modelo, y_cruda, y_mean, y_std, device='cpu'):
    """
    Recibe una señal física ruidosa e implementa la detección neuronal rápida.
    """
    modelo.eval() # Modo inferencia

    # Estandarizamos utilizando los valores guardados del entrenamiento
    y_normalizada = (y_cruda - y_mean) / y_std

    # Convertimos a tensor
    tensor_entrada = torch.tensor([[y_normalizada]], dtype=torch.float32).to(device)

    with torch.no_grad():
        p_hat = modelo(tensor_entrada).item()

    # Regla de decisión neuronal estricta
    simbolo_detectado = 2.0 if p_hat >= 0.5 else -2.0

    return simbolo_detectado, p_hat

# ==========================================
# 6. EJECUCIÓN PRINCIPAL
# ==========================================
if __name__ == "__main__":
    # Configuración base
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Dispositivo activo: {device}\n")

    # Simulación del canal de entrenamiento
    N_SAMPLES = 100000
    NOISE_VAR = 1.5

    print("1. Generando datos y calculando parámetros estadísticos...")
    y_data, t_data = generate_communications_data(N_SAMPLES, NOISE_VAR)

    # AQUÍ CAPTURAMOS y_mean_train y y_std_train AUTOMÁTICAMENTE
    train_dl, val_dl, test_dl, y_mean_train, y_std_train = prepare_dataloaders(y_data, t_data, batch_size=128)

    print("2. Entrenando el Detector Neuronal...")
    detector_neuronal = DetectorMLP(hidden_neurons=16).to(device)
    detector_neuronal = train_model(detector_neuronal, train_dl, val_dl, epochs=20, learning_rate=0.005, device=device)
    print("Entrenamiento finalizado y pesos óptimos restaurados.\n")

    # --- PRUEBA EN TIEMPO REAL (PRODUCCIÓN) ---
    print("==================================================")
    print("📡 FASE DE PRODUCCIÓN: RECEPCIÓN EN TIEMPO REAL 📡")
    print("==================================================")

    # Imaginemos que llegan 3 señales ruidosas por la antena
    observaciones_recibidas = [-1.85, 0.45, 2.10]

    for obs in observaciones_recibidas:
        simbolo, probabilidad = detectar_simbolo_en_tiempo_real(
            modelo=detector_neuronal,
            y_cruda=obs,
            y_mean=y_mean_train, # Se pasa el valor exacto del dataset
            y_std=y_std_train,   # Se pasa el valor exacto del dataset
            device=device
        )

        print(f"Voltaje recibido del canal: {obs:5.2f} V")
        print(f" -> Probabilidad Neuronal (de ser un +2): {probabilidad*100:6.2f}%")
        print(f" -> Decisión del Sistema: Símbolo {simbolo}")
        print("-" * 50)

Dispositivo activo: cpu

1. Generando datos y calculando parámetros estadísticos...
2. Entrenando el Detector Neuronal...
Entrenamiento finalizado y pesos óptimos restaurados.

📡 FASE DE PRODUCCIÓN: RECEPCIÓN EN TIEMPO REAL 📡
Voltaje recibido del canal: -1.85 V
 -> Probabilidad Neuronal (de ser un +2):   0.72%
 -> Decisión del Sistema: Símbolo -2.0
--------------------------------------------------
Voltaje recibido del canal:  0.45 V
 -> Probabilidad Neuronal (de ser un +2):  76.77%
 -> Decisión del Sistema: Símbolo 2.0
--------------------------------------------------
Voltaje recibido del canal:  2.10 V
 -> Probabilidad Neuronal (de ser un +2):  99.56%
 -> Decisión del Sistema: Símbolo 2.0
--------------------------------------------------
